# PyTorch Notes, Part 2: Batching → `nn.Module` → Training Loop

Continues from [`pytorch_notes.ipynb`](pytorch_notes.ipynb) (Part 1: Tensors, Autograd, core tensor ops) — starts at assembling a batch with `cat`/`stack`, through building a network with `nn.Module`, `Dataset`/`DataLoader`, the training loop, device placement, and saving/loading.

## `cat` vs `stack`: assembling a batch

Both combine tensors — the difference is whether a *new* dimension is created. `cat` joins
along an *existing* dimension (no new axis); `stack` creates a *new* dimension and lines
tensors up along it. This is the actual mechanism behind turning individual examples into a
batch: a `DataLoader` collating N separate `(3, 224, 224)` images into one `(N, 3, 224, 224)`
batch tensor is `torch.stack`, not `torch.cat`.

In [ ]:
img1 = torch.zeros(3, 4, 4)   # pretend: one 3-channel, 4x4 image
img2 = torch.ones(3, 4, 4)    # another one

print("cat dim=0 shape:  ", torch.cat([img1, img2], dim=0).shape)     # (6, 4, 4) — no new axis, just longer
print("stack dim=0 shape:", torch.stack([img1, img2], dim=0).shape)   # (2, 3, 4, 4) — new "batch" axis

# this is exactly how a batch of N images gets built from a list of individual images:
batch = torch.stack([img1, img2])
print("\nbatch of 2 images:", batch.shape, "-> (batch_size, channels, height, width)")

cat dim=0 shape:   torch.Size([6, 4, 4])
stack dim=0 shape: torch.Size([2, 3, 4, 4])

batch of 2 images: torch.Size([2, 3, 4, 4]) -> (batch_size, channels, height, width)


## Indexing and slicing — same rules as NumPy

Used constantly in ML code: pulling a single example out of a batch, grabbing one channel
out of an image, slicing a window out of a sequence.

In [ ]:
print("whole batch:", batch.shape)
print("first image only, batch[0]:", batch[0].shape)          # drop the batch dim
print("first channel of every image, batch[:, 0]:", batch[:, 0].shape)
print("a 2x2 crop of the first image, batch[0, :, :2, :2]:", batch[0, :, :2, :2].shape)

whole batch: torch.Size([2, 3, 4, 4])
first image only, batch[0]: torch.Size([3, 4, 4])
first channel of every image, batch[:, 0]: torch.Size([2, 4, 4])
a 2x2 crop of the first image, batch[0, :, :2, :2]: torch.Size([3, 2, 2])


# From Autograd to a Real Network

We now have the two ingredients autograd needs: tensors that carry `requires_grad=True`,
and operations that build a graph. But so far every example has been 2-3 loose tensors
(`a`, `b`, `x`) we tracked by hand. **A real network has thousands to billions of these —
manually naming and tracking each one doesn't scale.** That's the gap the next brick fills.

## Building a layer, by hand first

We already wrote `y = xW + b` using `@`. Let's actually run it with real, gradient-tracked
weights — then see what PyTorch gives us for free once we stop doing this by hand.

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0]])
w = torch.randn(3, 4, requires_grad=True)
b = torch.randn(4, requires_grad=True)

y_manual = x @ w + b
print("manual y:", y_manual)

y_manual.sum().backward()
print("w.grad shape:", w.grad.shape, "b.grad shape:", b.grad.shape)
# this works, but imagine doing this for 50 layers — 100 separate w/b tensors to name,
# initialize, and pass to an optimizer by hand. that's the actual problem nn.Module solves.

manual y: tensor([[-0.5782,  0.7264, -0.1513, -3.0938]], grad_fn=<AddBackward0>)
w.grad shape: torch.Size([3, 4]) b.grad shape: torch.Size([4])


## `nn.Linear`: the same math, but PyTorch owns the bookkeeping

`nn.Linear(in_features, out_features)` creates and tracks the exact `w`/`b` pair we just
built by hand — same `y = xW + b`, same `requires_grad=True`, but now the weights live
*inside an object* PyTorch knows how to find, move to a device, and hand to an optimizer.

In [ ]:
import torch.nn as nn

layer = nn.Linear(3, 4)

# prove it's doing the exact same math — copy our manual weights in and compare
with torch.no_grad():
    layer.weight.copy_(w.t())   # nn.Linear stores weight as (out_features, in_features) — transposed vs our manual w
    layer.bias.copy_(b)

y_module = layer(x)
print("nn.Linear y:", y_module)
print("matches manual y:", torch.allclose(y_manual, y_module))
print("\nlayer.weight.shape:", layer.weight.shape, "<- (out_features, in_features), not (in, out)")

nn.Linear y: tensor([[-0.5782,  0.7264, -0.1513, -3.0938]], grad_fn=<AddmmBackward0>)
matches manual y: True

layer.weight.shape: torch.Size([4, 3]) <- (out_features, in_features), not (in, out)


## Why stacking `nn.Linear` layers alone doesn't help

Natural next question: if one `nn.Linear` is a layer, does stacking several make a
"deeper," more powerful network? **No — not without something nonlinear between them.**
`y = W2(W1x + b1) + b2` algebraically collapses to `y = (W2W1)x + (W2b1 + b2)` — still just
one linear transform, `W_combined x + b_combined`. Provable, not just asserted:

In [ ]:
layer1 = nn.Linear(3, 5)
layer2 = nn.Linear(5, 2)
data = torch.randn(4, 3)

stacked_out = layer2(layer1(data))

# collapse the two layers into one equivalent linear transform, algebraically
W_combined = layer2.weight @ layer1.weight
b_combined = layer2.weight @ layer1.bias + layer2.bias
collapsed_out = data @ W_combined.t() + b_combined

print("2 stacked linear layers == 1 collapsed linear layer:",
      torch.allclose(stacked_out, collapsed_out, atol=1e-6))

# now put a ReLU between them
out_with_relu = layer2(torch.relu(layer1(data)))
print("same trick, WITH relu in between:",
      torch.allclose(out_with_relu, collapsed_out, atol=1e-6), "<- breaks. ReLU is what makes depth matter.")

2 stacked linear layers == 1 collapsed linear layer: True
same trick, WITH relu in between: False <- breaks. ReLU is what makes depth matter.


## A multilayer network (MLP): `nn.Module` subclass

`nn.Module` is the base class for a whole network, not just one layer — subclass it,
register layers in `__init__` (assigning `self.fc1 = nn.Linear(...)` auto-registers it,
since `nn.Module` overrides `__setattr__` to recognize sub-layers), define the forward
pass in `forward()`. This is what "the network" actually means in code.

In [ ]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 8)
        self.fc2 = nn.Linear(8, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))   # <- the nonlinearity from the cell above, now doing real work
        return self.fc2(x)

model = TinyNet()
print("parameters PyTorch found automatically, just from self.fc1/self.fc2 assignment:")
for name, p in model.named_parameters():
    print(" ", name, tuple(p.shape))

parameters PyTorch found automatically, just from self.fc1/self.fc2 assignment:
  fc1.weight (8, 3)
  fc1.bias (8,)
  fc2.weight (1, 8)
  fc2.bias (1,)


## Dataset + DataLoader: before we can train, we need batches

The model takes a batch of examples in one forward call (we already know why — `cat`
vs `stack` earlier is literally how a batch gets assembled). `Dataset` just needs to know
"how many examples" (`__len__`) and "give me example i" (`__getitem__`); `DataLoader` does
the batching, shuffling, and — for a real dataset — the parallel loading.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# toy regression target: y = sum(x) + a little noise
X = torch.randn(50, 3)
y = X.sum(dim=1, keepdim=True) + torch.randn(50, 1) * 0.1

dataset = ToyDataset(X, y)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

for i, (xb, yb) in enumerate(loader):
    print(f"batch {i}: xb.shape={xb.shape} yb.shape={yb.shape}")
# 50 / 16 -> 3 full batches + 1 short batch of 2. the last batch being a different size
# is normal — worth knowing before it surprises you inside a training loop.

batch 0: xb.shape=torch.Size([16, 3]) yb.shape=torch.Size([16, 1])
batch 1: xb.shape=torch.Size([16, 3]) yb.shape=torch.Size([16, 1])
batch 2: xb.shape=torch.Size([16, 3]) yb.shape=torch.Size([16, 1])
batch 3: xb.shape=torch.Size([2, 3]) yb.shape=torch.Size([2, 1])


## Loss: the model makes predictions, but "wrong how much"?

Autograd can compute gradients of *anything* scalar — but "anything" needs to actually be
defined. Loss is that definition: a single number saying how wrong the current predictions
are. No loss, no `.backward()` target, no gradient, no learning.

## Optimizer: autograd computes the gradient, it doesn't apply it

`loss.backward()` fills in `.grad` on every parameter. Nothing has changed the weights yet
— that's the optimizer's one job: read `.grad`, apply an update rule (`AdamW` here — same
decoupled-weight-decay optimizer as everywhere else in this workspace), and only then do
the weights actually move.

## The training loop: every brick above, in one cycle

`zero_grad → forward → loss → backward → step`, repeated. This is the entire mechanism —
every bigger model in this workspace runs this exact five-line cycle, just with more
layers and a fancier loss.

![The 5-step deep learning recipe: prediction → loss → gradient → update → reset](torch_5_step.png)

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)

losses = []
for epoch in range(30):
    for xb, yb in loader:                # each epoch = one full pass over the DataLoader
        optimizer.zero_grad()            # clear grads from the last step
        pred = model(xb)                 # forward pass
        loss = loss_fn(pred, yb)         # how wrong right now
        loss.backward()                  # compute gradients
        optimizer.step()                 # actually update the weights
        losses.append(loss.item())

print("first batch loss:", losses[0])
print("last batch loss: ", losses[-1])

first batch loss: 2.205120086669922
last batch loss:  0.020950492471456528


## Device placement: why `.to(device)` at all

The loop above ran entirely on CPU — fine for 50 toy examples, not fine once a model has
millions of parameters and data has millions of rows. A GPU (or Apple Silicon's MPS) does
the same matmuls in parallel across thousands of cores instead of one at a time. The rule
that actually matters: **the model and the data it's fed must live on the same device** —
mixing a CPU model with a GPU tensor (or vice versa) is a real, common error, not a
performance nitpick.

In [ ]:
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model.to(device)          # moves every parameter to the device, in place
X_dev = X.to(device)      # data has to move too — separately, it's not automatic

pred = model(X_dev)
print("prediction lives on:", pred.device)

device: mps


prediction lives on: mps:0


## Saving and loading: training is expensive, don't redo it

`state_dict()` is just the plain dict of every parameter tensor (we already met this on
`nn.Module` — same mechanism). Save that dict, not the model object itself; reload it into
a fresh model instance with the same architecture.

In [ ]:
save_path = "/tmp/tiny_net_demo.pt"   # a real project would use its own checkpoints/ dir
torch.save(model.state_dict(), save_path)

fresh_model = TinyNet().to(device)          # a NEW, randomly-initialized instance
fresh_model.load_state_dict(torch.load(save_path))
fresh_model.eval()

with torch.no_grad():
    same = torch.allclose(model(X_dev), fresh_model(X_dev))
print("reloaded model gives identical predictions to the trained one:", same)

reloaded model gives identical predictions to the trained one: True


## Where to keep going

Every brick is now in place: Tensors (data) → Autograd (gradients) → Operations (math) →
`nn.Module` (organized parameters) → Loss + Optimizer (learning signal + weight updates) →
DataLoader + training loop (the repeatable cycle) → device + save/load (making it real).

The next natural question the video likely asks: what changes when the model is too big
for one GPU, or training needs to run across several? That's covered hands-on, with real
measured numbers on this same machine, in
[`mini-llms-playground/from_scratch/tinystories-gpt-6m/docs/EFFICIENT_TRAINING.md`](../../mini-llms-playground/from_scratch/tinystories-gpt-6m/docs/EFFICIENT_TRAINING.md)
(mixed precision, gradient checkpointing) and
[`.../docs/DISTRIBUTED_TRAINING.md`](../../mini-llms-playground/from_scratch/tinystories-gpt-6m/docs/DISTRIBUTED_TRAINING.md)
(DDP/FSDP) — the exact same `zero_grad → forward → loss → backward → step` loop from this
notebook, just with more machinery wrapped around it.